# Event-rate and extinction maps

This notebook computes line-of-sight microlensing optical-depth and event-rate maps with apparent source selections in two bands:

- `I`: isochrone source selection, `14 <= I_S <= 21`
- `H`: isochrone source selection, `11 <= H_S <= 18`

Both selections use the genstars `E(J-Ks)` extinction map. The rate summaries are returned directly from the C++ simulation state; they are not parsed from CLI stdout. See `docs/rate_maps.md` for the API overview.


In [ ]:
from pathlib import Path
import os
import sys
from contextlib import contextmanager

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Find build/ whether Jupyter is launched from the repository root or examples/.
cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "examples" else cwd
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens

pd.set_option("display.max_columns", 80)
repo_root


## Configuration

The grid below is intentionally broad. Both band maps use the same isochrone source-selection machinery so their source densities and event rates are comparable. These cells can take a while because each sightline builds an isochrone selected-source density grid.


In [ ]:
N_SIMU = 1_000
SEED = 2026
EXTINCTION_LAW = 1  # 0: Alonso-Garcia+17, 1: Nishiyama+09, 2: Wang & Chen 2019
EXTINCTION_MAP = 1  # bundled genstars low-resolution E(J-Ks) map with subgrid values

# User-facing extinction adjustment knobs. The effective map value is
# E(J-Ks)_eff = E(J-Ks)_map * EJK_SCALE + EJK_OFFSET.
EJK_SCALE = 1.0
EJK_OFFSET = 0.0

l_values = np.linspace(-5.0, 5.0, 21)
b_values = np.linspace(-6.0, 6.0, 26)

BANDS = {
    "I": {
        "label": "I band",
        "min_mag": 14.0,
        "max_mag": 21.0,
        "mode": "isochrone",
        "band": "Imag",
        "photometry": "prime",
        "selection_label": r"$14 \leq I_S \leq 21$",
        "extinction_attr": "Imag",
        "extinction_label": r"$A_{I,RC}$ [mag]",
    },
    "H": {
        "label": "H band",
        "min_mag": 11.0,
        "max_mag": 18.0,
        "mode": "isochrone",
        "band": "Hmag_2mass",
        "photometry": "prime",
        "selection_label": r"$11 \leq H_S \leq 18$",
        "extinction_attr": "Hmag",
        "extinction_label": r"$A_{H,RC}$ [mag]",
    },
}


def effective_ejk(ejk_map_value):
    return ejk_map_value * EJK_SCALE + EJK_OFFSET


## Extinction maps

`GenstarsExtinctionMap` returns `E(J-Ks)` at each sightline. `genstars_reference_extinction` converts the effective `E(J-Ks)` into red-clump band extinctions for plotting and for the apparent source selections.


In [ ]:
extinction_map = genulens.GenstarsExtinctionMap.load_default(extinction_map=EXTINCTION_MAP)

ejk_map_grid = np.array([
    [extinction_map.ejk_at(l, b, extinction_map=EXTINCTION_MAP) for l in l_values]
    for b in b_values
])
ejk_grid = effective_ejk(ejk_map_grid)


def extinction_grid_for_band(band_key):
    attr = BANDS[band_key]["extinction_attr"]
    return np.array([
        [getattr(genulens.genstars_reference_extinction(l, b, ejk_grid[ib, il], EXTINCTION_LAW), attr)
         for il, l in enumerate(l_values)]
        for ib, b in enumerate(b_values)
    ])

extinction_grids = {band_key: extinction_grid_for_band(band_key) for band_key in BANDS}

pd.DataFrame(
    {
        "EJK_map_min": [np.nanmin(ejk_map_grid)],
        "EJK_map_max": [np.nanmax(ejk_map_grid)],
        "EJK_eff_min": [np.nanmin(ejk_grid)],
        "EJK_eff_max": [np.nanmax(ejk_grid)],
        "AIrc_min": [np.nanmin(extinction_grids["I"])],
        "AIrc_max": [np.nanmax(extinction_grids["I"])],
        "AHrc_min": [np.nanmin(extinction_grids["H"])],
        "AHrc_max": [np.nanmax(extinction_grids["H"])],
    }
)


## Rate summaries

Build one config per sightline so that both `l,b` and the corresponding extinction map value enter the selected-source density and event-rate calculation.


In [ ]:
@contextmanager
def suppress_native_stderr():
    """Temporarily hide C/C++ stderr messages in this notebook cell."""
    sys.stderr.flush()
    saved_stderr_fd = os.dup(2)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), 2)
            yield
    finally:
        sys.stderr.flush()
        os.dup2(saved_stderr_fd, 2)
        os.close(saved_stderr_fd)


def config_for_sightline(l_deg, b_deg, band_key):
    spec = BANDS[band_key]
    cfg = genulens.Config(l=float(l_deg), b=float(b_deg), n_simu=N_SIMU, seed=SEED)
    if spec["mode"] == "classic":
        cfg.use_classic_source(i_min=spec["min_mag"], i_max=spec["max_mag"])
    elif spec["mode"] == "isochrone":
        cfg.use_isochrone_source(
            i_min=spec["min_mag"],
            i_max=spec["max_mag"],
            band=spec["band"],
            photometry=spec["photometry"],
            apparent=True,
            min_mass=0.09,
            max_mass=2.0,
        )
    else:
        raise ValueError(spec["mode"])

    cfg.use_genstars_extinction_map(
        extinction_law=EXTINCTION_LAW,
        extinction_map=EXTINCTION_MAP,
        ejk_scale=EJK_SCALE,
        ejk_offset=EJK_OFFSET,
    )
    # Keep this rate example focused on source selection and event-rate summaries.
    cfg.observation.IL_err = 0.0
    return cfg


RATE_SUMMARY_COLUMNS = [
    "l",
    "b",
    "n_simu",
    "n_generated",
    "n_like",
    "source_density_arcmin2",
    "source_density_raw_arcmin2",
    "tau",
    "mean_tE_days",
    "median_tE_days",
    "event_rate_per_star_per_year",
    "event_rate_per_deg2_per_year",
    "sum_gamma",
    "sum_tE_gamma",
]


def rate_summary_row(summary):
    return {column: getattr(summary, column) for column in RATE_SUMMARY_COLUMNS}


def compute_rate_summaries_with_progress(configs, label):
    total = len(configs)
    rows = []
    print(
        f"Computing {label} rate map for {total} sightlines. "
        "This can take a while; detailed native progress logs are hidden in this notebook."
    )
    iterator = configs
    if tqdm is not None:
        iterator = tqdm(configs, total=total, desc=f"{label} sightlines", unit="sightline")
    with suppress_native_stderr():
        for cfg in iterator:
            rows.append(rate_summary_row(genulens.compute_rate_summary(cfg)))
    return pd.DataFrame(rows, columns=RATE_SUMMARY_COLUMNS)


def compute_band_rate_map(band_key):
    spec = BANDS[band_key]
    configs = [config_for_sightline(l, b, band_key) for b in b_values for l in l_values]
    if spec["mode"] == "isochrone":
        df = compute_rate_summaries_with_progress(configs, spec["label"])
    else:
        print(f"Computing {spec['label']} rate map.")
        result = genulens.compute_rate_summaries(configs)
        df = pd.DataFrame(result.to_numpy(), columns=result.columns)
    df["band"] = band_key
    return df

rate_maps = {band_key: compute_band_rate_map(band_key) for band_key in BANDS}
rate_maps["I"].head()


## Four-panel maps

Each band shows the band extinction map, optical depth, event rate per source star, and event rate per square degree.


In [ ]:
def grid_from_column(df, column):
    return df.pivot(index="b", columns="l", values=column).sort_index().to_numpy()


def positive_for_log(values):
    values = np.asarray(values, dtype=float)
    return np.where(values > 0.0, values, np.nan)


def plot_four_panel_map(band_key):
    spec = BANDS[band_key]
    df = rate_maps[band_key]
    tau_grid = positive_for_log(grid_from_column(df, "tau"))
    rate_star_grid = positive_for_log(grid_from_column(df, "event_rate_per_star_per_year"))
    rate_deg2_grid = positive_for_log(grid_from_column(df, "event_rate_per_deg2_per_year"))

    fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
    fig.suptitle(f"{spec['label']} source selection: {spec['selection_label']}")

    # Reverse the longitude axis so Galactic longitude increases to the left.
    extent = [l_values.max(), l_values.min(), b_values.min(), b_values.max()]

    im0 = axes[0, 0].imshow(extinction_grids[band_key], origin="lower", aspect="auto", extent=extent)
    axes[0, 0].set_title("Band extinction")
    fig.colorbar(im0, ax=axes[0, 0], label=spec["extinction_label"])

    im1 = axes[0, 1].imshow(
        tau_grid,
        origin="lower",
        aspect="auto",
        extent=extent,
        norm=LogNorm(vmin=np.nanmin(tau_grid), vmax=np.nanmax(tau_grid)),
    )
    axes[0, 1].set_title("Optical depth")
    fig.colorbar(im1, ax=axes[0, 1], label="tau")

    im2 = axes[1, 0].imshow(
        rate_star_grid,
        origin="lower",
        aspect="auto",
        extent=extent,
        norm=LogNorm(vmin=np.nanmin(rate_star_grid), vmax=np.nanmax(rate_star_grid)),
    )
    axes[1, 0].set_title("Event rate per source star")
    fig.colorbar(im2, ax=axes[1, 0], label="events / star / yr")

    im3 = axes[1, 1].imshow(
        rate_deg2_grid,
        origin="lower",
        aspect="auto",
        extent=extent,
        norm=LogNorm(vmin=np.nanmin(rate_deg2_grid), vmax=np.nanmax(rate_deg2_grid)),
    )
    axes[1, 1].set_title("Event rate per square degree")
    fig.colorbar(im3, ax=axes[1, 1], label="events / deg$^2$ / yr")

    for ax in axes.flat:
        ax.set_xlabel("l [deg]")
        ax.set_ylabel("b [deg]")

    return fig

for band_key in BANDS:
    plot_four_panel_map(band_key)
    plt.show()


## Latitude profiles integrated over longitude

The profiles below integrate over `-2 <= l <= 2` and show the result as a function of `b`.

For `tau` and event rate per source star, the profile is source-density weighted over longitude. For event rate per square degree, the profile is the longitude integral of the map value, so the unit is events per year per degree in latitude.


In [ ]:
def latitude_profile(df, l_min=-2.0, l_max=2.0):
    selected = df[(df["l"] >= l_min) & (df["l"] <= l_max)].copy()
    if len(selected) == 0:
        raise ValueError("empty longitude selection")

    l_unique = np.sort(selected["l"].unique())
    dl = float(np.median(np.diff(l_unique))) if len(l_unique) > 1 else 1.0

    rows = []
    for b, group in selected.groupby("b", sort=True):
        source_weight = group["source_density_arcmin2"].to_numpy()
        source_weight_sum = source_weight.sum()
        rows.append(
            {
                "b": b,
                "tau": np.average(group["tau"], weights=source_weight),
                "event_rate_per_star_per_year": np.average(
                    group["event_rate_per_star_per_year"], weights=source_weight
                ),
                "event_rate_per_deg2_integrated_per_year_per_deg_b": (
                    group["event_rate_per_deg2_per_year"].sum() * dl
                ),
                "source_density_arcmin2_mean": group["source_density_arcmin2"].mean(),
                "source_density_arcmin2_integrated_deg_l": group["source_density_arcmin2"].sum() * dl,
                "source_weight_sum": source_weight_sum,
            }
        )
    return pd.DataFrame(rows)

profiles = {band_key: latitude_profile(rate_maps[band_key]) for band_key in BANDS}
profiles["I"].head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

for band_key, profile in profiles.items():
    label = BANDS[band_key]["label"]
    axes[0].plot(profile["b"], profile["tau"], marker="o", label=label)
    axes[1].plot(profile["b"], profile["event_rate_per_star_per_year"], marker="o", label=label)
    axes[2].plot(
        profile["b"],
        profile["event_rate_per_deg2_integrated_per_year_per_deg_b"],
        marker="o",
        label=label,
    )

axes[0].set_title(r"$-2 \leq l \leq 2$: optical depth")
axes[0].set_xlabel("b [deg]")
axes[0].set_ylabel("source-weighted tau")
axes[0].set_yscale("log")

axes[1].set_title(r"$-2 \leq l \leq 2$: event rate per source")
axes[1].set_xlabel("b [deg]")
axes[1].set_ylabel("events / star / yr")
axes[1].set_yscale("log")

axes[2].set_title(r"$-2 \leq l \leq 2$: event rate per area")
axes[2].set_xlabel("b [deg]")
axes[2].set_ylabel("events / yr / deg in b")
axes[2].set_yscale("log")

for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()

plt.show()


## Notes

- The event rate follows the existing CLI convention: `2 * tau / pi / mean_tE_days * 365.25`.
- `event_rate_per_star_per_year` is the per-source-star rate.
- `event_rate_per_deg2_per_year` additionally multiplies by the selected source surface density.
- I-band uses the legacy luminosity-function source selection.
- H-band uses the isochrone selected-source density grid, so it is slower but conditions the event-rate calculation on the requested H-band magnitude range.
- Use larger `N_SIMU` for smoother event-rate maps and profiles.
